In [2]:
f_keys = []
g_keys = []
with open("handout/output.txt", "r") as file:
    for line in file:
        key, value = line.strip().split(" = ", 1)
        if key == "cipher_f":
            cipher_f = bytes.fromhex(value)
        if key == "cipher_g":
            cipher_g = bytes.fromhex(value)
        if key.startswith("fk"):
            f_keys.append(bytes.fromhex(value))
        if key.startswith("gk"):
            g_keys.append(bytes.fromhex(value))

n = len(cipher_f)
R = len(f_keys)

In [3]:
from handout.chall import xor_bytes, f, g

def F(x: bytes) -> bytes:
    for i in range(R):
        x = f(x)
        x = xor_bytes(x, f_keys[i])
    return x


def G(x: bytes) -> bytes:
    for i in range(R):
        x = g(x)
        x = xor_bytes(x, g_keys[i])
    return x

In [9]:
GF2 = GF(2)

def bytes_to_gf2(b: bytes) -> vector:
    bits = [(byte >> j) & 1 for byte in b for j in range(8)]
    return vector(GF2, flatten(bits))

def gf2_to_bytes(v) -> bytes:
    bits = [v[i : i + 8] for i in range(0, len(v), 8)]
    return bytes([int(ZZ(list(bits), base=2)) for bits in bits])

def get_matrix(func, dim):
    A = []
    for i in range(dim):
        input_vector = gf2_to_bytes(
            vector(GF2, [0 if j != i else 1 for j in range(dim)])
        )
        output_vector = bytes_to_gf2(func(input_vector))
        A.append(output_vector)
    return matrix(GF2, A).T

In [ ]:
A = get_matrix(lambda x: F(x) + G(x), n * 8)
y = bytes_to_gf2(cipher_f + cipher_g)
x = A.solve_right(y)
gf2_to_bytes(x)